# ProofAgent™ — Log-based (Colab)

Needs a **Log-Based** project key. `pip install proofagent-sdk` · **`PROOFAGENT_API_KEY`** in Secrets.

Same naming as **`examples/judge_led_quickstart.py`** where applicable: **`tested_agent_config`**, **`your_agent`**, **`byo`**, **`pa`**, **`result`**, **`logs`**, and **`print_full_evaluation_report`**.

**Role** slug must match your project (default **`customer_support`**).

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "proofagent-sdk"])

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["PROOFAGENT_API_KEY"] = userdata.get("PROOFAGENT_API_KEY")
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
except ImportError:
    pass
assert os.environ.get("PROOFAGENT_API_KEY", "").strip()

In [ ]:
import os

from proofagent import ProofAgent, TestedAgent, print_full_evaluation_report

logs = [
    {"turn_index": 1, "user_message": "Hi", "agent_answer": "Hello."},
    {"turn_index": 2, "user_message": "Refund?", "agent_answer": "I can help."},
]

tested_agent_config = {
    "role": "customer_support",
    "description": (
        "IT operations and support agent: triages incidents, looks up ticket status, and answers user "
        "questions with concise, policy-aware replies. Aligns with the LangGraph ReAct IT-ops example."
    ),
    "tools": [
        {"name": "ticket_stub", "description": "Look up ticket status (demo / ITSM integration)."},
    ],
}

your_agent = TestedAgent.from_json(tested_agent_config)
byo = os.environ.get("OPENAI_API_KEY", "").strip()


async def main():
    async with ProofAgent.from_env(
        reasoning_provider="openai" if byo else None,
        reasoning_model="gpt-4o-mini" if byo else None,
    ) as pa:
        result = await pa.evaluate_logs(
            logs,
            your_agent,
            verbose=True,
            poll_complete_timeout=900.0,
        )
        print(result.label, result.score, result.run_id)
        print("\n--- Run ID ---\n", result.run_id)
        print_full_evaluation_report(result.report)


await main()